# Data Freeze Filter — Visit Date Range

This notebook takes UDS data freeze CSV files and filters them down to visits that fall within a specified date range. Anything outside that window gets dropped, and the cleaned files are saved and ready to use.

---

### What it does
1. Loads all UDS data CSVs from `input/fw_downloads/` (or a single file if you prefer)
2. Reads the visit date column in each file and checks each row against your date range
3. Keeps only visits that fall between `START_DATE` and `END_DATE` (both dates inclusive)
4. Exports the filtered files to `output/uds_data/` with clean, readable filenames

---

### Required folder structure

```
datafreeze_ETL_scripts/
├── input/
│   ├── dd_downloads/              <- data dictionary CSV files
│   └── fw_downloads/              <- place UDS data CSV files here
├── output/
│   ├── data_dictionary/           <- output from data_dictionary.ipynb
│   └── uds_data/                  <- output from this notebook (auto-created)
└── notebooks/
    └── data_freeze_filter.ipynb   <- you are here
```

---

### Two ways to run

| Mode | When to use |
|---|---|
| `"folder"` | Process every CSV in `fw_downloads/` at once — the most common use case |
| `"standalone"` | Process a single file — handy for a quick check or one-off submission |

Start by filling in your date range and run mode in the **Configuration** cell below.

In [6]:
# Standard libraries used throughout this notebook.
# pathlib keeps file paths clean and consistent across operating systems.
# re is used to strip date codes out of filenames on export.
import re
from pathlib import Path

import pandas as pd

## Configuration

Fill in the two cells below before running anything else.

- **Run mode** — choose whether to process all files at once or just one
- **Date range** — set the start and end dates for the visits you want to keep
- **Paths** — these should work as-is if your folder structure matches the diagram above

In [7]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# "folder"     -> picks up every CSV in fw_downloads/ automatically
# "standalone" -> processes only the file named in SINGLE_FILE below
MODE        = "folder"
SINGLE_FILE = "south-texas-adrc-uds-2026-04-15T20-32-36-538Z.csv"  # only used when MODE = "standalone"

# ── Date range ────────────────────────────────────────────────────────────────
# Visits on or between these two dates will be kept. Everything outside the
# window is dropped. Both START_DATE and END_DATE are inclusive.
# Format: "YYYY-MM-DD"
START_DATE = "2019-01-01"
END_DATE   = "2024-12-31"

# ── Visit date column ─────────────────────────────────────────────────────────
# The name of the date column in your files. Update this if your files use
# a different column name.
VISIT_DATE_COL = "visitdate"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Leave these as-is if your folder structure matches the diagram in the intro.
# Only change them if you've moved files around.
DOWNLOADS_DIR = Path("../input/fw_downloads")
OUTPUT_DIR    = Path("../output/uds_data")

## Load the files

Scans the downloads folder and loads every CSV it finds into memory. Each file gets its own dataframe so they can be processed independently.

In [8]:
# Folder mode picks up every CSV in fw_downloads/ without you needing to
# know the exact filenames — useful since downloaded files have a date stamp
# appended that changes with each new data freeze.
# Standalone mode loads just the one file you named above.
if MODE == "standalone":
    path = DOWNLOADS_DIR / SINGLE_FILE
    dataframes = {path.name: pd.read_csv(path)}
else:
    dataframes = {
        f.name: pd.read_csv(f)
        for f in sorted(DOWNLOADS_DIR.glob("*.csv"))
    }

# A quick row and column count per file — good to glance at before filtering
# to make sure everything loaded as expected.
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

## Filter by visit date

Applies the date range from the Configuration cell to every loaded file. Each file gets its own before/after row count so you can see exactly how much was trimmed.

In [9]:
def filter_by_visit_date(df, name):
    # If the expected date column isn't there, skip filtering for that file
    # and print a heads-up rather than throwing an error and stopping everything.
    if VISIT_DATE_COL not in df.columns:
        print(f"Heads up: '{VISIT_DATE_COL}' column not found in {name} — this file was not filtered.")
        return df

    # Convert the date column to proper datetime values so comparisons work correctly.
    # Anything that can't be parsed (blank cells, bad formatting) becomes a null
    # value and gets excluded — the message below makes that transparent.
    df[VISIT_DATE_COL] = pd.to_datetime(df[VISIT_DATE_COL], errors="coerce")
    n_invalid = df[VISIT_DATE_COL].isna().sum()
    if n_invalid:
        print(f"Heads up: {name} has {n_invalid} rows with unreadable dates — these were excluded.")

    # Keep only rows that fall within the date range (both ends included).
    return df[
        (df[VISIT_DATE_COL] >= pd.Timestamp(START_DATE)) &
        (df[VISIT_DATE_COL] <= pd.Timestamp(END_DATE))
    ]

result = {}
for name, df in dataframes.items():
    filtered = filter_by_visit_date(df, name)
    dropped  = len(df) - len(filtered)
    result[name] = filtered
    print(f"{name}: {len(df)} rows total -> {len(filtered)} kept, {dropped} dropped")

dataframes = result

## Export the filtered files

Saves each filtered dataframe as a CSV in `output/uds_data/`. The date stamp that gets appended to downloaded filenames is stripped out so the exported names stay short and consistent across freeze versions.

In [10]:
# Creates the output folder if it isn't there yet — no need to create it manually.
OUTPUT_DIR.mkdir(exist_ok=True)

def clean_filename(name):
    # Downloaded files come with a date stamp in the name
    # (e.g. "south-texas-adrc-uds-2026-04-15T20-32-36-538Z.csv").
    # We strip that out here so the saved file has a clean, stable name
    # that won't be different the next time a new freeze is downloaded.
    stem = Path(name).stem
    stem = re.sub(r"\s*\(\d+\)", "", stem)   # remove " (1)" from accidental duplicate downloads
    stem = re.sub(r"-ded-\d+", "", stem)         # remove "-ded-MMDDYYYY" style date suffixes
    stem = re.sub(r"T\d{2}-\d{2}-\d{2}-\d+Z$", "", stem)  # remove ISO timestamp suffixes
    return stem + ".csv"

for name, df in dataframes.items():
    out_name = clean_filename(name)
    df.to_csv(OUTPUT_DIR / out_name, index=False)
    print(f"{name}  ->  {out_name}")